<a href="https://colab.research.google.com/github/muhammadabdurrehmanmaqsood/flyrank-ml-internship-abdurrehman/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadabdurrehmanmaqsood/flyrank-ml-internship-abdurrehman/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [12]:
#**Unit of Analysis:** One row represents one content item's daily performance
#for a specific client (report_date × client_id × content_id).

#**Time Window:** A single mid-panel month, strictly covering March 1, 2026,
#through March 31, 2026 (month=2026-03). We avoid the final month (_sample) to
#preserve our sealed testing environment.

In [13]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import login

# Authenticate and load the mid-panel month
login(token=userdata.get('HF_TOKEN'))
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/*"
)
df = dataset['train'].to_pandas()

# Verify time window
min_date = df['report_date'].min()
max_date = df['report_date'].max()
print(f"Time Window Verified: {min_date} to {max_date}")

Time Window Verified: 2026-03-01 to 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [14]:
#**Features:**
#**ctr_prev30:** Knowable at the decision moment because it relies entirely on
#the trailing 30-day historical window.
#**avg_position_prev30:** Knowable at the decision moment because search
#position is recorded prior to the prediction date.
#**word_count:** Knowable at the decision moment as it is a static property of
#the content upon publication.
#**has_keyword_data:** Knowable at the decision moment; flags missingness to
#avoid injecting category signals.
#**ai_traffic_pct:** Knowable at the decision moment because traffic sources are
#logged historically (values can exceed 100 due to mismatched measurement
#systems).

#**Label:** is_declining_label — the target we are attempting to predict.

#**Context:** client_id, content_id, report_date — used exclusively for grouping
#and splitting, never as model features.

#**Excluded:** trend_direction and trend_pct — Excluded because
#is_declining_label is mathematically derived directly from them. Including them
#leaks the answer to the model.

In [15]:
# THE LEAKAGE TRAP EXPERIMENT
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Inspect what we actually have
print("Available warehouse columns:", df.columns.tolist()[:10], "...")

# 2. Create a proxy label: Did it rank on page 1? (Position 1-10)
# Remember the gotcha: avg_position = 0 means "no data", not rank 0
df['is_page_one'] = (df['gsc_avg_position'] <= 10) & (df['gsc_avg_position'] > 0)

# 3. Create a deliberate leakage feature (a fake metric derived from the label)
df['leaked_rank_score'] = np.where(df['is_page_one'], 100, 10)

# 4. Simulate a small feature frame (dropping missing targets for a clean test)
model_df = df.dropna(subset=['is_page_one', 'gsc_avg_position', 'ga4_data_available', 'leaked_rank_score']).copy()

# --- WITH THE TRAP (Leakage) ---
leaked_features = ['ga4_data_available', 'leaked_rank_score']
X_leak = model_df[leaked_features]
y = model_df['is_page_one']

X_train, X_test, y_train, y_test = train_test_split(X_leak, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
leaked_score = accuracy_score(y_test, clf.predict(X_test))
print(f"Accuracy WITH leakage trap (leaked_rank_score included): {leaked_score:.4f}")

# --- HONEST BASELINE (Trap Removed) ---
honest_features = ['ga4_data_available'] # A safe, knowable context feature
X_honest = model_df[honest_features]

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_honest, y, test_size=0.2, random_state=42)
clf_honest = RandomForestClassifier(max_depth=3, random_state=42).fit(X_train_h, y_train_h)
honest_score = accuracy_score(y_test_h, clf_honest.predict(X_test_h))
print(f"Accuracy WITHOUT leakage trap (honest baseline): {honest_score:.4f}")

Available warehouse columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position'] ...
Accuracy WITH leakage trap (leaked_rank_score included): 1.0000
Accuracy WITHOUT leakage trap (honest baseline): 0.5256


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:
#The queries below definitively prove our contract claims. The grain holds
#perfectly (max rows per grouping is 1), the row count aligns with our monthly
#slice expectations, and strict boolean filtering reveals the true availability
#of analytics data.

In [17]:
# 1. Print columns to verify the exact names
print("Available columns in this slice:", df.columns.tolist()[:15])

# 1. Verify Grain
# Using the exact hash IDs revealed by the column printout
grain_check = df.groupby(['report_date', 'client_hash_id', 'content_hash_id']).size()
print(f"\nMax rows per grain grouping: {grain_check.max()} (Expected: 1)")

# 2. Row Counts
print(f"Total rows in slice: {len(df):,}")

# 3. Availability (Strict IS TRUE filtering)
# Using == True to mimic SQL's 'IS TRUE', discarding both False and NULL
if 'ga4_data_available' in df.columns:
    ga4_available_count = len(df[df['ga4_data_available'] == True])
    null_ga4_count = df['ga4_data_available'].isna().sum()
    print(f"Rows with usable GA4 data (IS TRUE): {ga4_available_count:,}")
    print(f"Rows with NULL GA4 data (Not measured): {null_ga4_count:,}")
else:
    print("Warning: 'ga4_data_available' not found.")

# 4. Missingness Pattern
# Checking a known warehouse metric
if 'gsc_avg_position' in df.columns:
    missing_position = df['gsc_avg_position'].isna().mean() * 100
    print(f"Percentage of missing gsc_avg_position data: {missing_position:.1f}%")

Available columns in this slice: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions']

Max rows per grain grouping: 1 (Expected: 1)
Total rows in slice: 9,841,378
Rows with usable GA4 data (IS TRUE): 413,966
Rows with NULL GA4 data (Not measured): 3,018,741
Percentage of missing gsc_avg_position data: 63.3%


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [18]:
#This data cannot provide a uniform historical baseline across all clients.
#Because history depth differs wildly per client, defining a single global
#calendar window will silently fail. Furthermore, early rows in a client's
#history may have ga4_data_available as strictly NULL; these missing values mean
#the analytics were not yet measured, not that the content had zero engagement.

In [19]:
# Check disparate start dates per client to prove the limitation
if 'gsc_data_start' in df.columns:
    client_starts = df.groupby('client_id')['gsc_data_start'].min()
    print("Sample of differing client start dates:")
    print(client_starts.head())
else:
    print("Note: gsc_data_start lives in dim_clients, requiring a join to fully map history depth.")

Note: gsc_data_start lives in dim_clients, requiring a join to fully map history depth.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.